In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
llm = ChatOllama(model="gemma4:e2b", base_url="http://127.0.0.1:11434")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈어요! 😊\n\n저는 사용자님을 돕기 위해 항상 준비하고 있답니다. 사용자님은 잘 지내셨어요?', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:50:29.9690804Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25493297400, 'load_duration': 24876373100, 'prompt_eval_count': 21, 'prompt_eval_duration': 161662000, 'eval_count': 33, 'eval_duration': 449096000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7e0-f73a-7a51-9637-4c411e72c30a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 33, 'total_tokens': 54})

In [12]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [13]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [14]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:50:34.15267Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4151143300, 'load_duration': 7107300, 'prompt_eval_count': 157, 'prompt_eval_duration': 582060000, 'eval_count': 342, 'eval_duration': 3556354000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7e1-5af0-7d90-9d2f-b0a020b51586-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '09fc9383-66c2-4707-9a90-f7163f83709c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 342, 'total_tokens': 499})]


In [15]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': 'Busan', 'timezone': 'Asia/Seoul'}
Asia/Seoul (Busan) 현재시각 2026-09-22 15:50:34 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:50:34.15267Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4151143300, 'load_duration': 7107300, 'prompt_eval_count': 157, 'prompt_eval_duration': 582060000, 'eval_count': 342, 'eval_duration': 3556354000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7e1-5af0-7d90-9d2f-b0a020b51586-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '09fc9383-66c2-4707-9a90-f7163f83709c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 342, 'total_tokens': 499}),
 ToolMessage(content='Asia/Seoul (Busan) 현재시각 2026-09-22 15:50:34 ', name='

In [16]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 **2026년 9월 22일 15시 50분 34초**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:50:35.4365372Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1268749000, 'load_duration': 5296300, 'prompt_eval_count': 227, 'prompt_eval_duration': 70493000, 'eval_count': 126, 'eval_duration': 1183606000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7e1-6b36-7843-bb0f-c21289dc6955-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 126, 'total_tokens': 353})

In [17]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")


In [18]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수
    Args:
        stock_history_input.ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
        stock_history_input.period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")
    """
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [19]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={} response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:50:48.5397881Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4832777500, 'load_duration': 6479400, 'prompt_eval_count': 389, 'prompt_eval_duration': 736437000, 'eval_count': 390, 'eval_duration': 4074894000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'} id='lc_run--01a0c7e1-9079-7a51-86d7-0d665ae7d2b7-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}, 'id': '921ba91c-08f2-4695-814f-e0b17435dfb6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 389, 'output_tokens': 390, 'total_tokens': 779}


In [20]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-08-24 00:00:00-04:00 | 361.41 | 363.24 | 348.26 |  348.95 | 3.91905e+07 |           0 |              0 |\n| 2026-08-25 00:00:00-04:00 | 349.58 | 357    | 349.2  |  350.25 | 2.98167e+07 |           0 |              0 |\n| 2026-08-26 00:00:00-04:00 | 345.27 | 351.93 | 342.53 |  345.82 | 2.8573e+07  |           0 |              0 |\n| 2026-08-27 00:00:00-04:00 | 346.16 | 355.74 | 345.45 |  354.81 | 3.01362e+07 |           0 |              0 |\n| 2026-08-28 00:00:00-04:00 | 357.1  | 358.8  | 345.2  |  348.75 | 3.29722e+07 |           0 |              0 |\n| 2026-08-31 00:00:00-04:00 | 347.21 | 368.92 | 347.15 |  367.95 | 6.17323e+07 |           0 |              0 |\n| 2026-09-01 00:00:00-04:0

In [21]:
llm_with_tools.invoke(messages)

AIMessage(content='제공해 주신 데이터를 기준으로 볼 때, 테슬라(TSLA) 주가는 한 달 전보다 **올랐습니다**.\n\n(제공된 데이터 기간: 2026년 8월 24일부터 2026년 9월 21일까지)', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T06:51:14.1535054Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7651111300, 'load_duration': 6498000, 'prompt_eval_count': 2074, 'prompt_eval_duration': 755347000, 'eval_count': 663, 'eval_duration': 6876180000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c7e1-e984-7282-b00e-16d7254aa214-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2074, 'output_tokens': 663, 'total_tokens': 2737})